# DevMind AI — Dataset Selection, Inspection & Download

This notebook turns candidate Hugging Face datasets into a reviewed shortlist. **Do not download large datasets blindly.** Check dataset cards, licenses, schemas, and samples first.

In [ ]:
!pip -q install datasets huggingface_hub pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, pandas as pd
PROJECT_DIR = Path('/content/drive/MyDrive/DevMindAI')
RAW_DIR = PROJECT_DIR / 'datasets' / 'raw'
META_DIR = PROJECT_DIR / 'datasets' / 'metadata'
SELECTED_DIR = PROJECT_DIR / 'datasets' / 'selected'
for p in [RAW_DIR, META_DIR, SELECTED_DIR]: p.mkdir(parents=True, exist_ok=True)
print('DevMind AI folders ready:', PROJECT_DIR)

## 1. Candidate shortlist
The search results from the previous notebook are candidates only. Add dataset IDs from those results here after reviewing their cards. The starter list below includes well-known code/debugging dataset candidates; verify current licenses and terms before use.

In [ ]:
# Replace/add IDs based on the dataset search results and card review.
CANDIDATE_DATASETS = [
    # 'owner/dataset_name',
    'google-research-datasets/mbpp',
    'bigcode/the-stack-smol',
    'codeparrot/github-code',
]
print('Candidates:', CANDIDATE_DATASETS)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
records = []
for dataset_id in CANDIDATE_DATASETS:
    try:
        info = api.dataset_info(dataset_id)
        card = info.cardData or {}
        records.append({'dataset_id': dataset_id, 'downloads': info.downloads, 'likes': info.likes,
                       'license': card.get('license', 'NOT DECLARED'),
                       'tags': ', '.join(info.tags[:12]), 'last_modified': str(info.last_modified),
                       'card_url': f'https://huggingface.co/datasets/{dataset_id}'})
    except Exception as e:
        records.append({'dataset_id': dataset_id, 'error': str(e)})
candidate_df = pd.DataFrame(records)
display(candidate_df)
candidate_df.to_csv(META_DIR / 'candidate_metadata.csv', index=False)
print('Saved metadata:', META_DIR / 'candidate_metadata.csv')

## 2. License and quality review
Use this checklist for every candidate. A missing or unclear license means **do not use for training until clarified**.

- [ ] License explicitly reviewed
- [ ] Dataset card and intended-use terms reviewed
- [ ] Schema inspected
- [ ] Sample records reviewed
- [ ] Programming/debugging relevance confirmed
- [ ] Personal data/secrets risk assessed
- [ ] Duplicate/contamination risk assessed
- [ ] Dataset size and Colab storage/GPU feasibility assessed

In [ ]:
# Inspect a dataset's schema and a few samples WITHOUT downloading the full dataset.
from datasets import load_dataset

INSPECT_DATASET_ID = ''  # Enter one reviewed dataset ID
INSPECT_CONFIG = None    # Set config if required
INSPECT_SPLIT = 'train'

if INSPECT_DATASET_ID:
    try:
        ds = load_dataset(INSPECT_DATASET_ID, name=INSPECT_CONFIG, split=f'{INSPECT_SPLIT}[:5]', streaming=True)
        sample_rows = list(ds)
        print('Sample count:', len(sample_rows))
        display(pd.DataFrame(sample_rows))
    except Exception as e:
        print('Inspection error:', repr(e))
else:
    print('Set INSPECT_DATASET_ID after reviewing the dataset card.')

## 3. Download only an approved dataset
Use a small split first. Avoid full downloads until size, license, schema, and sample quality are confirmed.

In [ ]:
APPROVED_DATASET_ID = ''  # Fill only after review
APPROVED_CONFIG = None
APPROVED_SPLIT = 'train'
MAX_EXAMPLES = 5000  # Pilot cap; increase only after validation

if APPROVED_DATASET_ID:
    ds = load_dataset(APPROVED_DATASET_ID, name=APPROVED_CONFIG, split=f'{APPROVED_SPLIT}[:{MAX_EXAMPLES}]')
    out_path = RAW_DIR / (APPROVED_DATASET_ID.replace('/', '__') + '.jsonl')
    ds.to_json(str(out_path))
    print('Downloaded pilot:', len(ds), 'examples')
    print('Saved to:', out_path)
else:
    print('No approved dataset selected. Nothing downloaded.')

## 4. Record dataset provenance
Create a record for every downloaded dataset. Keep source ID, date, license, split, subset size, and any transformations.

In [ ]:
provenance = {
    'dataset_id': APPROVED_DATASET_ID,
    'config': APPROVED_CONFIG,
    'split': APPROVED_SPLIT,
    'max_examples': MAX_EXAMPLES,
    'download_date_utc': pd.Timestamp.utcnow().isoformat(),
    'license_reviewed': False,
    'dataset_card_reviewed': False,
    'sample_quality_reviewed': False,
    'notes': 'Update this record after manual review.'
}
if APPROVED_DATASET_ID:
    p = META_DIR / (APPROVED_DATASET_ID.replace('/', '__') + '_provenance.json')
    p.write_text(json.dumps(provenance, indent=2))
    print('Provenance saved:', p)
else:
    print('Select an approved dataset before recording provenance.')

## Next
After collecting a few reviewed pilot datasets, we will build **Notebook 04: Data Cleaning, Normalization, Deduplication & Train/Validation/Test Splits**.